In [2]:
from concurrent.futures import ProcessPoolExecutor

pool = ProcessPoolExecutor()

def p_fib(n):
    if n <= 1:
        return n

    future_x = pool.submit(p_fib, n - 1)
    y = p_fib(n - 2)
    x = future_x.result()

    return x + y

In [1]:
"""
Sequential
"""
def mat_vec(A, x):
    n = len(A)
    y = [0] * n

    for i in range(n):
        for j in range(n):
            y[i] += A[i][j] * x[j]

    return y

"""
Parallel
"""
from concurrent.futures import ProcessPoolExecutor

def row_dot(args):
    row, x = args
    return sum(row[j] * x[j] for j in range(len(x)))

def p_mat_vec(A, x):
    with ProcessPoolExecutor() as executor:
        y = list(executor.map(row_dot, [(row, x) for row in A]))
    return y

In [3]:
from concurrent.futures import ProcessPoolExecutor
import multiprocessing

def compute_row(A, x, i):
    total = 0
    for j in range(len(x)):
        total += A[i][j] * x[j]
    return i, total

def p_mat_recursive(A, x, y, i, I, executor):
    if i == I:
        row_index, value = compute_row(A, x, i)
        y[row_index] = value
        return

    mid = (i + I) // 2

    # spawn left half
    future = executor.submit(
        p_mat_recursive,
        A, x, y,
        i, mid,
        executor
    )

    p_mat_recursive(A, x, y, mid + 1, I, executor)
    future.result()

if __name__ == "__main__":
    n = 4

    A = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 14, 15, 16]
    ]

    x = [1, 1, 1, 1]

    # Shared output vector
    manager = multiprocessing.Manager()
    y = manager.list([0] * n)

    with ProcessPoolExecutor() as executor:
        p_mat_recursive(A, x, y, 0, n - 1, executor)

    print(list(y))

In [ ]:
"""
Simplier numpy example:
"""
import numpy as np

A = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
])

x = np.array([1, 1, 1, 1])

y = A @ x

print(y)

In [4]:
"""
T_P = T_1 / P + T_∞

T_P = running time on P processors
T_1 = work (total time on 1 processor)
P = number of processors
T_∞ = span / critical-path length (minimum possible execution time with infinitely many processors)

T_1 = 15
P = 3
T_∞ = 5

T_P ≤ 15/3 + 5
T_P ≤ 5 + 5
T_P ≤ 10

The work term: T_1 / P

If total work is 15 and you have 3 processors:
15 / 3 = 5

meaning each processor ideally does 5 units of work.

The span term: T_∞

This is the unavoidable sequential dependency chain.
Even with infinitely many processors, you still cannot go faster than the longest chain of dependent operations.
For p_fib(5):

fib(5) → fib(4) → fib(3) → fib(2) → fib(1)
which gives span 5.

Running Time ≈ (work per processor) + (unavoidable sequential time)
"""

'\nT_P = T_1 / P + T_∞\n\nT_P = running time on P processors\nT_1 = work (total time on 1 processor)\nP = number of processors\nT_∞ = span / critical-path length (minimum possible execution time with infinitely many processors)\n\nT_1 = 15\nP = 3\nT_∞ = 5\n\nT_P ≤ 15/3 + 5\nT_P ≤ 5 + 5\nT_P ≤ 10\n\nThe work term: T_1 / P\n\nIf total work is 15 and you have 3 processors:\n15 / 3 = 5\n\nmeaning each processor ideally does 5 units of work.\n\nThe span term: T_∞\n\nThis is the unavoidable sequential dependency chain.\nEven with infinitely many processors, you still cannot go faster than the longest chain of dependent operations.\nFor p_fib(5):\n\nfib(5) → fib(4) → fib(3) → fib(2) → fib(1)\nwhich gives span 5.\n\nRunning Time ≈ (work per processor) + (unavoidable sequential time)\n'

In [5]:
"""
In a greedy schedule:

if at least P strands are ready, all processors stay busy,
otherwise, fewer than P strands are ready.

The key observation:
Any step with fewer than P ready strands must execute a strand on the critical path.
Because if a critical-path strand were not executed, then at least one processor could execute it, contradicting greediness.

Thus:
every "incomplete" step reduces the remaining span by 1,
and there can be at most T_∞ such steps.

The standard bound:
T_P ≤ T_1/P + T_∞
double-counts some work on the critical path.

The stronger bound removes that overlap:
T_P ≤ (T_1 - T_∞)/P + T_∞

T_∞ units of work are already "reserved" for the critical path,
only the remaining work must be divided among processors.

So the improved bound is slightly tighter.

======================================================================================================================================

A "good" greedy scheduler always advances the critical path immediately.
A "bad" greedy scheduler repeatedly uses processors on side work first, delaying critical-path progress by almost one extra step per level.

Fast schedule ≈ T∞
Slow schedule ≈ 2T∞

Greedy scheduling guarantees processors never sit idle unnecessarily, but it does not guarantee that the scheduler chooses the "best" ready strands.
"""

'\nIn a greedy schedule:\n\nif at least P strands are ready, all processors stay busy,\notherwise, fewer than P strands are ready.\n\nThe key observation:\nAny step with fewer than P ready strands must execute a strand on the critical path.\nBecause if a critical-path strand were not executed, then at least one processor could execute it, contradicting greediness.\n\nThus:\nevery "incomplete" step reduces the remaining span by 1,\nand there can be at most T_∞ such steps.\n\nThe standard bound:\nT_P ≤ T_1/P + T_∞\ndouble-counts some work on the critical path.\n\nThe stronger bound removes that overlap:\nT_P ≤ (T_1 - T_∞)/P + T_∞\n\nT_∞ units of work are already "reserved" for the critical path,\nonly the remaining work must be divided among processors.\n\nSo the improved bound is slightly tighter.\n\n======================================================================================================================================\n\nA "good" greedy scheduler always advances the criti

In [6]:
"""
Given measurements:

T_4 = 80
T_10 = 42
T_64 = 10

Meaning:

T_4 = 80
→ the program took 80 seconds when executed on 4 processors.

T_10 = 42
→ the program took 42 seconds when executed on 10 processors.

T_64 = 10
→ the program took 10 seconds when executed on 64 processors.

These values are execution times measured for the SAME parallel algorithm using different numbers of processors. As the number of processors increases, the running time should generally decrease because more work can be done simultaneously.

Using the Work Law:

T_P ≥ T_1 / P

we estimate the total work T_1. Using the 4-processor run:

80 ≥ T_1 / 4

which gives:

T_1 ≤ 320

This means the total amount of work done by the algorithm is at most 320 units of time.

Using the Span Law:

T_P ≥ T_∞

and the 64-processor run:

10 ≥ T_∞

which gives:

T_∞ ≤ 10

This means the critical path (the longest chain of dependent operations that cannot run in parallel) is at most 10 time units.

Finally, using the greedy scheduler bound:

T_P ≤ (T_1 - T_∞)/P + T_∞

for P = 10:

T_10 ≤ (320 - 10)/10 + 10
T_10 ≤ 31 + 10
T_10 ≤ 41

But the measurements claim:

T_10 = 42

which violates the theoretical upper bound. Therefore, the three timing measurements cannot all be correct simultaneously.
"""

'\nGiven measurements:\n\nT_4 = 80\nT_10 = 42\nT_64 = 10\n\nMeaning:\n\nT_4 = 80\n→ the program took 80 seconds when executed on 4 processors.\n\nT_10 = 42\n→ the program took 42 seconds when executed on 10 processors.\n\nT_64 = 10\n→ the program took 10 seconds when executed on 64 processors.\n\nThese values are execution times measured for the SAME parallel algorithm using different numbers of processors. As the number of processors increases, the running time should generally decrease because more work can be done simultaneously.\n\nUsing the Work Law:\n\nT_P ≥ T_1 / P\n\nwe estimate the total work T_1. Using the 4-processor run:\n\n80 ≥ T_1 / 4\n\nwhich gives:\n\nT_1 ≤ 320\n\nThis means the total amount of work done by the algorithm is at most 320 units of time.\n\nUsing the Span Law:\n\nT_P ≥ T_∞\n\nand the 64-processor run:\n\n10 ≥ T_∞\n\nwhich gives:\n\nT_∞ ≤ 10\n\nThis means the critical path (the longest chain of dependent operations that cannot run in parallel) is at most 10 

In [7]:
"""
parallel for i = 1 to n
    parallel for j = 1 to n
        p[j] = A[i][j] * x[j]

    y[i] = parallel_reduce_sum(p[1..n])

T_1/T_∞ = Θ(n^2)/Θ(lgn)

Parallelism = Θ(n^2 / lg n)

| Quantity    | Value         |
| ----------- | --------------|
| Work        | Θ(n^2)        |
| Span        | Θ(lgn)        |
| Parallelism | Θ(n^2 / lg n) |

========================================================================================================

Given the modified algorithm:
def p_transpose(A, n):
    for j = 2 to n
        parallel for i = 1 to j-1
            exchange A[i][j] with A[j][i]

the total work T_1 is unchanged because the algorithm still performs the same number of swaps. The total number of exchanges is:

Σ(j−1) from j=2 to n = 1 + 2 + ... + (n−1) = n(n−1)/2

therefore:

T_1 = Θ(n²)

The span changes because the outer loop is now serial. For each value of j, the inner parallel loop has span Θ(lg j). Since the outer loop executes sequentially, the spans add together:

T_∞ = ΣΘ(lg j) from j=2 to n

which simplifies using logarithm summation:

T_∞ = Θ(lg(n!)) = Θ(n lg n)

Finally, the parallelism is:

T_1 / T_∞ = Θ(n²) / Θ(n lg n) = Θ(n / lg n)

Therefore, the modified algorithm has:
Work: T_1 = Θ(n²)
Span: T_∞ = Θ(n lg n)
Parallelism: Θ(n / lg n)

The key difference is that only the inner loop remains parallel, while the outer loop forces the computation to proceed column-by-column sequentially, increasing the critical-path length substantially.

========================================================================================================

For the fully parallel version of p_transpose, the running time is:
T_P^(1) = n²/P + lg n

because the work is T_1 = Θ(n²) and the span is T_∞ = Θ(lg n). For the modified version with a serial outer loop, the running time is:
T_P^(2) = n²/P + n lg n

because the work remains T_1 = Θ(n²) but the span increases to T_∞ = Θ(n lg n). To determine when the two algorithms run equally fast, set the running times equal:
n²/P + lg n = n²/P + n lg n

Subtracting n²/P from both sides gives:
lg n = n lg n

Dividing both sides by lg n yields:
1 = n

Therefore, the two versions are equally fast only when n = 1. For any practical matrix size n > 1, the fully parallel version is always faster because both algorithms perform the same total work, but the second version has a much larger span, which increases the critical-path length and limits parallel speedup.
"""

'\nparallel for i = 1 to n\n    parallel for j = 1 to n\n        p[j] = A[i][j] * x[j]\n\n    y[i] = parallel_reduce_sum(p[1..n])\n\nT_1/T_∞ = Θ(n^2)/Θ(lgn)\n\nParallelism = Θ(n^2 / lg n)\n\n| Quantity    | Value         |\n| ----------- | --------------|\n| Work        | Θ(n^2)        |\n| Span        | Θ(lgn)        |\n| Parallelism | Θ(n^2 / lg n) |\n\n========================================================================================================\n\nGiven the modified algorithm:\ndef p_transpose(A, n):\n    for j = 2 to n\n        parallel for i = 1 to j-1\n            exchange A[i][j] with A[j][i]\n\nthe total work T_1 is unchanged because the algorithm still performs the same number of swaps. The total number of exchanges is:\n\nΣ(j−1) from j=2 to n = 1 + 2 + ... + (n−1) = n(n−1)/2\n\ntherefore:\n\nT_1 = Θ(n²)\n\nThe span changes because the outer loop is now serial. For each value of j, the inner parallel loop has span Θ(lg j). Since the outer loop executes sequentially,